# Social Network Ads — Purchase Prediction

A reproducible classification project exploring whether age and salary are associated with a purchase decision. This notebook uses Logistic Regression and a held-out test set.

> This is a learning exercise on a small sample dataset, not evidence about real customers or a production model.

## 1. Setup

Install dependencies with `pip install -r requirements.txt`. The CSV is stored at `data/Social_Network_Ads.csv` relative to this notebook.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

DATA_PATH = Path("data") / "Social_Network_Ads.csv"
df = pd.read_csv(DATA_PATH)
df.head()

## 2. Inspect the data

Review the available fields, data types, missing values, and class balance before modeling.

In [ ]:
print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")
display(df.dtypes.to_frame("dtype"))
display(df.isna().sum().to_frame("missing values"))
display(df["Purchased"].value_counts().rename(index={0: "Not purchased", 1: "Purchased"}))

## 3. Explore the features

Use `Age` and `Salary` as model inputs. `User ID` is an identifier and is excluded.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=df, x="Age", hue="Purchased", multiple="stack", ax=axes[0])
axes[0].set_title("Age by purchase outcome")
sns.scatterplot(data=df, x="Age", y="Salary", hue="Purchased", alpha=0.75, ax=axes[1])
axes[1].set_title("Age and salary by purchase outcome")
plt.tight_layout()
plt.show()

## 4. Train the model

A stratified 70/30 split keeps class proportions similar across sets. The pipeline fits `StandardScaler` on training data only, then trains Logistic Regression.

In [ ]:
FEATURES = ["Age", "Salary"]
TARGET = "Purchased"
X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(random_state=42)),
])
model.fit(X_train, y_train)

## 5. Evaluate on held-out data

Review accuracy together with precision, recall, F1-score, and the confusion matrix.

In [ ]:
y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=["Not purchased", "Purchased"], zero_division=0))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["Not purchased", "Purchased"],
            yticklabels=["Not purchased", "Purchased"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion matrix")
plt.tight_layout()
plt.show()

## 6. Example prediction

This example is only an illustration of model output, not a reliable forecast for an individual.

In [ ]:
sample = pd.DataFrame([{"Age": 45, "Salary": 60000}])
prediction = model.predict(sample)[0]
probability = model.predict_proba(sample)[0, 1]
print(f"Predicted class: {prediction} (1 = purchased, 0 = not purchased)")
print(f"Estimated purchase probability: {probability:.1%}")

## Limitations

This small educational dataset does not establish causal relationships or real-world purchasing behavior. Results from a single train/test split can vary; consider cross-validation before drawing broader conclusions. Only `Age` and `Salary` are used as model features.